# Multilingual ASR with wav2vec2 XLS-R
### Fine-tuning Facebook XLS-R-300M on 5 Indian Languages + English

**Languages:** Hindi | Telugu | Tamil | Kannada | English

**Model:** facebook/wav2vec2-xls-r-300m | **Dataset:** Mozilla Common Voice 11.0

---
## Roadmap
- Phase 1: Setup (Cells 1-4)
- Phase 2: Hindi baseline (Cells 5-12)
- Phase 3: 5 languages (Cells 13-16)
- Phase 4: Evaluate + Hub (Cells 17-20)
- Phase 5: Gradio app (Cells 21-22)

> Colab tip: Runtime > Change runtime type > GPU (T4)

---
## Phase 1 - Setup

In [2]:
# Cell 1 - Install dependencies
!pip install -q transformers datasets torchaudio librosa jiwer \
    gradio huggingface_hub accelerate evaluate soundfile
print("All dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 108.9 MB/s eta 0:00:00
All dependencies installed!


In [3]:
# Cell 2 - Imports
import os, re, json
import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Union
import librosa
import soundfile as sf
import IPython.display as ipd
from datasets import load_dataset, concatenate_datasets, Audio, DatasetDict
from transformers import (
    Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor, Wav2Vec2ForCTC,
    TrainingArguments, Trainer,
)
import evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('No GPU! Go to Runtime > Change runtime type > GPU')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# Cell 3 - Login to HuggingFace
!hf auth login --token $HF_TOKEN

Hint: A new version of huggingface_hub (1.17.0) is available! You are using version 1.16.1.
To update, run: hf update
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Error: Invalid user token.
Set HF_DEBUG=1 as environment variable for full traceback.


In [ ]:
# Cell 4 - Configuration (UPDATED - using Google FLEURS dataset)
HF_USERNAME     = 'your_hf_username'  # Change this to your HuggingFace username
BASE_MODEL      = 'niranjan2026/xlsr-multilingual-in5'
DATASET_NAME    = 'google/fleurs'
MODEL_REPO_NAME = f'{HF_USERNAME}/xlsr-multilingual-in5'
SAMPLING_RATE   = 16_000

# FLEURS language codes (different from Common Voice)
LANGUAGES = [
    ('hi_in', 'Hindi',   'indic'),
    ('te_in', 'Telugu',  'indic'),
    ('ta_in', 'Tamil',   'indic'),
    ('kn_in', 'Kannada', 'indic'),
    ('en_us', 'English', 'latin'),
]

TRAIN_PER_LANG = 500   # FLEURS is smaller so we use 500
TEST_PER_LANG  = 100

print(f'Dataset : {DATASET_NAME}')
print(f'Model   : {MODEL_REPO_NAME}')
print(f'Languages: {[l[1] for l in LANGUAGES]}')

Dataset : google/fleurs
Model   : niranjan2026/xlsr-multilingual-in5
Languages: ['Hindi', 'Telugu', 'Tamil', 'Kannada', 'English']


---
## Phase 2 - Single-Language Baseline (Hindi)

**Why start with one language?**
- Debug the full pipeline quickly without running 5x the data
- Confirm everything works before the big multilingual run
- Establish a WER baseline to compare against

Hindi has the largest Indic representation in Common Voice.

In [15]:
# Cell 5 - Load Hindi from FLEURS (no login needed)

def load_split(lang_code, split, max_samples=None):
    ds = load_dataset(
        'google/fleurs',
        lang_code,
        split=split,
        trust_remote_code=False,
    )
    # FLEURS uses 'transcription' instead of 'sentence'
    # rename it so rest of the code stays the same
    ds = ds.rename_column('transcription', 'sentence')
    ds = ds.select_columns(['audio', 'sentence'])
    ds = ds.add_column('language', [lang_code] * len(ds))
    if max_samples and len(ds) > max_samples:
        ds = ds.select(range(max_samples))
    return ds


print('Loading Hindi from FLEURS...')
dataset = DatasetDict({
    'train': load_split('hi_in', 'train', TRAIN_PER_LANG),
    'test' : load_split('hi_in', 'test',  TEST_PER_LANG),
})
print(f'Train: {len(dataset["train"]):,}  |  Test: {len(dataset["test"]):,}')

Loading Hindi from FLEURS...
Train: 500  |  Test: 100


In [16]:
# Cell 6 - Explore and listen to a sample
sample = dataset['train'][0]
print(f'Language  : {sample["language"]}')
print(f'Transcript: {sample["sentence"]}')
print(f'Duration  : {len(sample["audio"]["array"]) / sample["audio"]["sampling_rate"]:.1f}s')
ipd.display(ipd.Audio(sample['audio']['array'], rate=sample['audio']['sampling_rate']))

Language  : hi_in
Transcript: राजनीतिज्ञों ने कहा कि उन्होंने निर्णायक मत को अनावश्यक रूप से निर्धारित करने के लिए अफ़गान संविधान में काफी अस्पष्टता पाई थी
Duration  : 8.6s


In [17]:
# Cell 7 - Text cleaning
# Remove digits/punctuation that CTC cannot model.
# Indic scripts are left unchanged. English is lowercased.

REMOVE_REGEX = r'[0-9\,\.\!\?\-\;\:\(\)\[\]\{\}\"\'\'\"\"\%\&\*\=\+\^\$\#\@\~\`\<\>\/\\|]'

def clean_text(batch):
    text = re.sub(REMOVE_REGEX, '', batch['sentence'])
    text = re.sub(r'\s+', ' ', text).strip()
    if batch.get('language') == 'en':
        text = text.lower()
    batch['sentence'] = text
    return batch

dataset = dataset.map(clean_text)
print('Cleaned samples:')
for i in range(3):
    s = dataset['train'][i]
    print(f'  [{s["language"]}] {s["sentence"]}')

Cleaned samples:
  [hi_in] राजनीतिज्ञों ने कहा कि उन्होंने निर्णायक मत को अनावश्यक रूप से निर्धारित करने के लिए अफ़गान संविधान में काफी अस्पष्टता पाई थी
  [hi_in] जब आप छुट्टी पर होते हैं तो आपके पास खुद के लिए समय होता है और कुछ विशेष करने के लिए अतिरिक्त समय निकालें
  [hi_in] वाइल्डलाइफ़ फ़ोटोग्राफ़ी को अक्सर हल्के में ही लिया जाता है लेकिन सामान्यतः फ़टोग्राफ़ी की तरह ही एक चित्र हज़ारों शब्द से अधिक कीमत रखता है


In [18]:
# Cell 8 - Build character vocabulary
# CTC models treat each CHARACTER as a token.
# For 5 languages the vocab will contain chars from multiple Unicode blocks.

def extract_all_chars(batch):
    all_text = ' '.join(batch['sentence'])
    return {'vocab': [list(set(all_text))], 'all_text': [all_text]}

vocabs = dataset.map(
    extract_all_chars, batched=True, batch_size=-1,
    keep_in_memory=True, remove_columns=dataset.column_names['train'],
)

vocab_set  = set(vocabs['train']['vocab'][0]) | set(vocabs['test']['vocab'][0])
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_set))}

# Space -> CTC word-boundary token
if ' ' in vocab_dict:
    vocab_dict['|'] = vocab_dict[' ']
    del vocab_dict[' ']
else:
    vocab_dict['|'] = len(vocab_dict)

vocab_dict['[UNK]'] = len(vocab_dict)
vocab_dict['[PAD]'] = len(vocab_dict)

os.makedirs('./vocab', exist_ok=True)
with open('./vocab/vocab.json', 'w', encoding='utf-8') as f:
    json.dump(vocab_dict, f, ensure_ascii=False, indent=2)

print(f'Vocabulary size: {len(vocab_dict):,}')

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Vocabulary size: 90


In [19]:
# Cell 9 - Build Tokenizer, Feature Extractor, Processor
# Tokenizer        : text -> token IDs (character-level)
# FeatureExtractor  : raw waveform -> normalised float32
# Processor         : combines both

tokenizer = Wav2Vec2CTCTokenizer(
    './vocab/vocab.json',
    unk_token='[UNK]', pad_token='[PAD]', word_delimiter_token='|',
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=SAMPLING_RATE,
    padding_value=0.0, do_normalize=True, return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor, tokenizer=tokenizer
)
processor.save_pretrained('./processor')
print(f'Processor saved. Vocab size: {tokenizer.vocab_size}')

# Quick sanity check
test_enc = tokenizer('hello')
print(f'Tokenizer test: hello -> {test_enc["input_ids"]}')

Processor saved. Vocab size: 90
Tokenizer test: hello -> [8, 5, 11, 11, 14]


In [20]:
# Cell 10 - Preprocess audio + text (FIXED for new Transformers version)

dataset = dataset.cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))

def preprocess_batch(batch):
    audio = batch['audio']

    # Process audio
    out = processor(
        audio['array'],
        sampling_rate=audio['sampling_rate'],
        return_attention_mask=True
    )
    batch['input_values']   = out.input_values[0]
    batch['attention_mask'] = out.attention_mask[0]

    # Process text — FIXED: use tokenizer directly instead of as_target_processor
    batch['labels'] = processor.tokenizer(batch['sentence']).input_ids

    return batch


print('Preprocessing audio + text...')
dataset = dataset.map(
    preprocess_batch,
    remove_columns=dataset.column_names['train'],
    num_proc=1
)
print(f'Done - {len(dataset["train"]):,} train | {len(dataset["test"]):,} test')

Preprocessing audio + text...
Done - 500 train | 100 test


In [12]:
# Cell 11 - Data Collator + WER metric (FIXED)

@dataclass
class DataCollatorCTCWithPadding:
    processor : Wav2Vec2Processor
    padding   : Union[bool, str] = True

    def __call__(self, features):
        inp = [{'input_values': f['input_values'], 'attention_mask': f['attention_mask']}
               for f in features]
        lbl = [{'input_ids': f['labels']} for f in features]

        # Pad audio
        batch = self.processor.pad(inp, padding=self.padding, return_tensors='pt')

        # Pad labels — FIXED: use tokenizer directly
        lb = self.processor.tokenizer.pad(
            lbl, padding=self.padding, return_tensors='pt'
        )
        batch['labels'] = lb['input_ids'].masked_fill(
            lb.attention_mask.ne(1), -100
        )
        return batch


data_collator = DataCollatorCTCWithPadding(processor=processor)
wer_metric    = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids  = np.argmax(pred.predictions, axis=-1)
    lbl       = pred.label_ids.copy()
    lbl[lbl == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids)
    label_str = processor.tokenizer.batch_decode(lbl, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    print(f'  WER: {wer*100:.2f}%')
    for p, l in zip(pred_str[:2], label_str[:2]):
        print(f'  REF: {l}  |  HYP: {p}')
    return {'wer': wer}


print('Collator and WER metric ready!')

Collator and WER metric ready!


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch, gc
gc.collect()
torch.cuda.empty_cache()

from transformers import Wav2Vec2ForCTC, TrainingArguments, Trainer

# Load model
model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.0,
    ctc_loss_reduction='mean',
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,
)
model.freeze_feature_encoder()
model = model.to('cuda')

print(f'Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

training_args = TrainingArguments(
    output_dir='./hindi-baseline',
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=3e-4,
    warmup_steps=200,
    num_train_epochs=30,           # increased from 5 to 30
    eval_strategy='steps',
    eval_steps=100,
    save_steps=100,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    fp16=True,
    group_by_length=True,
    push_to_hub=False,
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    processing_class=processor.feature_extractor,
)

print('Starting Hindi training (30 epochs)...')
print('WER will start dropping after ~500-600 steps')
print('='*50)
trainer.train()
print('Hindi training complete!')

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  WER: 49.88%
  REF: हालांकि लगभग रातोंरात इन योजनाओं को तब बनाया गया था जब सोवियत की यूनियन रेड आर्मी के से ज़्यादा सैनिकों ने पोलैंड के पूर्वी क्षेत्रों पर हमला करने के बाद प्रवेश करते हुए बेलारूसी और यूक्रेनी मोर्चे खड़े कर दिए थे उन्होंने यह रिगा शांति संधि द सोवियतपॉलिश नॉनएग्रेशन पैक्ट और अन्य अंतरराष्ट्रीय द्विपक्षीय और बहुपक्षीय दोनों तरह की संधियों का उल्लंघन करते हुए किया था  |  HYP: हालांग की लगव ग्रातु राथ इन योजनाओं को तब बनाया गया था ब सोविित की यूनियन रेडार्मी के आट लाक से ज़्याथा सेनिको ने पोलंड की पूर्विक ्षेत्रों बर हमलाकरने के बाद प्रवेश करते हुए बेलार उसी ओ जुक्रेनी मोर्चे कड़े कर दिए थए उन्होंने यए री गाशांति संधि सोवित पोलिश नॉनेगरेशनपैक्ट और अ्यअंतर राष्ट्रिद्वीर पक्षिय और बहु पक्षय दोनों तरह की संदियों काव लंघन गरती हुए किया था
  REF: पौधों से सबसे आसानी से मिलने वाले संसाधनों में पत्तियों और फलियों में मिलने वाले प्रोटीन होते लेकिन हम जैसे प्राइमेट्स के लिए उसे पकाए बिना पचाना मुश्किल होता है  |  HYP: पोधहसे सबसेर आसानहईं से मिलने बाले संसाधन में पत्तियो और फलय

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Hindi training complete!


---
## Phase 3 - Scale to 5 Languages

Same pipeline repeated for all 5 languages.
The multilingual vocab covers characters from Devanagari, Telugu, Tamil, Kannada, and Latin.

In [21]:
# Cell 13 - Load all 5 languages
print('Loading all 5 languages...')
tr_splits, te_splits = [], []
for code, name, _ in LANGUAGES:
    print(f'  {name} ({code})...', end=' ', flush=True)
    tr = load_split(code, 'train', TRAIN_PER_LANG)
    te = load_split(code, 'test',  TEST_PER_LANG)
    tr_splits.append(tr)
    te_splits.append(te)
    print(f'train={len(tr):,}  test={len(te):,}')

raw_train  = concatenate_datasets(tr_splits).shuffle(seed=42)
raw_test   = concatenate_datasets(te_splits)
ml_dataset = DatasetDict({'train': raw_train, 'test': raw_test})
print(f'Combined: {len(raw_train):,} train | {len(raw_test):,} test')

Loading all 5 languages...
  Hindi (hi_in)... train=500  test=100
  Telugu (te_in)... 

parquet-data/te_in/train-00000-of-00001.(…):   0%|          | 0.00/1.81G [00:00<?, ?B/s]

parquet-data/te_in/validation-00000-of-0(…):   0%|          | 0.00/205M [00:00<?, ?B/s]

parquet-data/te_in/test-00000-of-00001.p(…):   0%|          | 0.00/332M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2302 [00:00<?, ? examples/s]

...

parquet-data/en_us/test-00000-of-00001.p(…):   0%|          | 0.00/402M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2602 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/394 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/647 [00:00<?, ? examples/s]

train=500  test=100
Combined: 2,500 train | 500 test


In [22]:
# Cell 14 - Clean text + rebuild multilingual vocabulary
ml_dataset = ml_dataset.map(clean_text)

ml_vocabs = ml_dataset.map(
    extract_all_chars, batched=True, batch_size=-1,
    keep_in_memory=True, remove_columns=ml_dataset.column_names['train'],
)
ml_vocab_set  = set(ml_vocabs['train']['vocab'][0]) | set(ml_vocabs['test']['vocab'][0])
ml_vocab_dict = {v: k for k, v in enumerate(sorted(ml_vocab_set))}
if ' ' in ml_vocab_dict:
    ml_vocab_dict['|'] = ml_vocab_dict[' ']; del ml_vocab_dict[' ']
else:
    ml_vocab_dict['|'] = len(ml_vocab_dict)
ml_vocab_dict['[UNK]'] = len(ml_vocab_dict)
ml_vocab_dict['[PAD]'] = len(ml_vocab_dict)

os.makedirs('./vocab_ml', exist_ok=True)
with open('./vocab_ml/vocab.json', 'w', encoding='utf-8') as f:
    json.dump(ml_vocab_dict, f, ensure_ascii=False, indent=2)
print(f'Multilingual vocab: {len(ml_vocab_dict):,} tokens')

# Build multilingual processor
ml_tokenizer = Wav2Vec2CTCTokenizer(
    './vocab_ml/vocab.json',
    unk_token='[UNK]', pad_token='[PAD]', word_delimiter_token='|'
)
ml_processor = Wav2Vec2Processor(
    feature_extractor=Wav2Vec2FeatureExtractor(
        feature_size=1, sampling_rate=SAMPLING_RATE,
        padding_value=0.0, do_normalize=True, return_attention_mask=True
    ),
    tokenizer=ml_tokenizer,
)
ml_processor.save_pretrained('./processor_ml')
print('Multilingual processor saved to ./processor_ml')

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Multilingual vocab: 275 tokens
Multilingual processor saved to ./processor_ml


In [23]:
# Cell 15 - Preprocess multilingual dataset (FIXED)

ml_dataset = ml_dataset.cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))

def ml_preprocess(batch):
    audio = batch['audio']
    out   = ml_processor(
        audio['array'],
        sampling_rate=audio['sampling_rate'],
        return_attention_mask=True
    )
    batch['input_values']   = out.input_values[0]
    batch['attention_mask'] = out.attention_mask[0]

    # FIXED: use tokenizer directly
    batch['labels'] = ml_processor.tokenizer(batch['sentence']).input_ids

    return batch


print('Preprocessing 5 languages (5-10 min)...')
ml_dataset_proc = ml_dataset.map(
    ml_preprocess,
    remove_columns=ml_dataset.column_names['train'],
    num_proc=1
)
print(f'Done - Train: {len(ml_dataset_proc["train"]):,} | Test: {len(ml_dataset_proc["test"]):,}')

Preprocessing 5 languages (5-10 min)...


Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Done - Train: 2,500 | Test: 500


In [ ]:
# Run this first
from huggingface_hub import login

# Paste your NEW write token here
login(token='new_hf_write_token')
print("Done!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Done!


In [ ]:
import os

# Paste your new Write token here
new_token =   # ← paste real token

# Force set the token in environment
os.environ["HF_TOKEN"] = new_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = new_token

# Login with the new token
from huggingface_hub import login
login(token=new_token, add_to_git_credential=False)
print("Login done!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Login done!


In [ ]:
# Cell 16 - Multilingual Training

HF_USERNAME     = 'your_hf_username'  # Change this to your HuggingFace username
MODEL_REPO_NAME = f'{HF_USERNAME}/xlsr-multilingual-in5'

print(f'✅ Username   : {HF_USERNAME}')
print(f'✅ Model repo : {MODEL_REPO_NAME}')

# Clear memory
import os, gc, torch, numpy as np
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

gc.collect()
torch.cuda.empty_cache()

# Metrics + Collator
import evaluate

ml_collator = DataCollatorCTCWithPadding(processor=ml_processor)
ml_wer = evaluate.load("wer")

def ml_compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)

    labels = pred.label_ids.copy()
    labels[labels == -100] = ml_processor.tokenizer.pad_token_id

    pred_str = ml_processor.batch_decode(pred_ids)
    label_str = ml_processor.tokenizer.batch_decode(
        labels,
        group_tokens=False
    )

    wer = ml_wer.compute(
        predictions=pred_str,
        references=label_str
    )

    print(f"WER: {wer*100:.2f}%")

    return {"wer": wer}

# Load model from your saved HF checkpoint
from transformers import (
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer
)

ml_model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.0,
    ctc_loss_reduction="mean",
    pad_token_id=ml_processor.tokenizer.pad_token_id,
    vocab_size=len(ml_processor.tokenizer),
    ignore_mismatched_sizes=True,
)

ml_model.freeze_feature_encoder()
ml_model = ml_model.to("cuda")

print(f"✅ Free VRAM : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print("✅ Dataset   : google/fleurs")
print("✅ Languages : hi_in | te_in | ta_in | kn_in | en_us")

# Training arguments
ml_args = TrainingArguments(
    output_dir="./multilingual_training",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=3e-4,
    warmup_steps=300,
    num_train_epochs=30,

    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,

    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    fp16=True,
    group_by_length=True,

    # IMPORTANT
    push_to_hub=False,

    dataloader_num_workers=0,
)

ml_trainer = Trainer(
    model=ml_model,
    data_collator=ml_collator,
    args=ml_args,
    compute_metrics=ml_compute_metrics,
    train_dataset=ml_dataset_proc["train"],
    eval_dataset=ml_dataset_proc["test"],
    processing_class=ml_processor.feature_extractor,
)

print("\nStarting multilingual training...")
print("=" * 50)

ml_trainer.train()

print("\n✅ Training complete!")

✅ Username   : niranjan2026
✅ Model repo : niranjan2026/xlsr-multilingual-in5


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

✅ Free VRAM : 12.99 GB
✅ Dataset   : google/fleurs
✅ Languages : hi_in | te_in | ta_in | kn_in | en_us

Starting multilingual training...


Step,Training Loss,Validation Loss,Wer
100,67.567505,4.324260,1.000000


WER: 100.00%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import os

for root, dirs, files in os.walk("./multilingual_training"):
    if "checkpoint-100" in root:
        print(root)

In [ ]:
from transformers import TrainingArguments, Trainer
import torch, gc

gc.collect()
torch.cuda.empty_cache()

HF_USERNAME     = 'niranjan2026'
MODEL_REPO_NAME = f'{HF_USERNAME}/xlsr-multilingual-in5'

ml_args = TrainingArguments(
    output_dir=MODEL_REPO_NAME,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=3e-4,
    warmup_steps=300,
    num_train_epochs=30,
    eval_strategy='steps',
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    fp16=True,
    group_by_length=True,
    push_to_hub=True,
    hub_model_id=MODEL_REPO_NAME,
    hub_token=new_token,          # ← pass token directly
    dataloader_num_workers=0,
)

ml_trainer = Trainer(
    model=ml_model,
    data_collator=ml_collator,
    args=ml_args,
    compute_metrics=ml_compute_metrics,
    train_dataset=ml_dataset_proc['train'],
    eval_dataset=ml_dataset_proc['test'],
    processing_class=ml_processor.feature_extractor,
)

print('Starting multilingual training...')
print('='*50)
ml_trainer.train()
print('Done!')

In [ ]:
!ls /content/hindi-baseline

ls: cannot access '/content/hindi-baseline': No such file or directory


---
## Phase 4 - Evaluation and Hub Upload

In [ ]:
# Cell 17 - Overall evaluation
metrics = ml_trainer.evaluate()
print(f'Overall WER: {metrics["eval_wer"]*100:.2f}%')
print(f'Loss       : {metrics["eval_loss"]:.4f}')
ml_trainer.log_metrics('eval', metrics)
ml_trainer.save_metrics('eval', metrics)

In [ ]:
# Cell 18 - Per-language WER breakdown
print('Per-Language WER:')
for code, name, _ in LANGUAGES:
    idx = [i for i, s in enumerate(raw_test) if s['language'] == code][:50]
    if not idx: continue
    preds = ml_trainer.predict(ml_dataset_proc['test'].select(idx))
    p_ids = np.argmax(preds.predictions, axis=-1)
    l_ids = preds.label_ids.copy()
    l_ids[l_ids == -100] = ml_processor.tokenizer.pad_token_id
    wer   = ml_wer.compute(
        predictions=ml_processor.batch_decode(p_ids),
        references=ml_processor.batch_decode(l_ids, group_tokens=False)
    )
    print(f'  {name:10s} ({code}): {wer*100:.2f}%')

In [ ]:
# Cell 19 - Push to HuggingFace Hub
ml_trainer.push_to_hub(commit_message='Fine-tuned XLS-R multilingual IN5')
ml_processor.push_to_hub(MODEL_REPO_NAME)
print(f'Model live at: https://huggingface.co/{MODEL_REPO_NAME}')

In [ ]:
# Cell 20 - Inference on test samples
ml_model.to(device).eval()

def transcribe(arr, sr=SAMPLING_RATE):
    arr = arr.astype(np.float32)
    if arr.max() > 1.0: arr = arr / 32768.0
    if arr.ndim > 1: arr = arr.mean(axis=1)
    if sr != SAMPLING_RATE:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLING_RATE)
    inp = ml_processor(arr, sampling_rate=SAMPLING_RATE, return_tensors='pt', padding=True)
    with torch.no_grad():
        logits = ml_model(**{k: v.to(device) for k, v in inp.items()}).logits
    return ml_processor.batch_decode(torch.argmax(logits, dim=-1))[0]


print('Inference tests:')
seen = set()
for s in raw_test:
    lang = s['language']
    if lang in seen: continue
    result = transcribe(s['audio']['array'], s['audio']['sampling_rate'])
    print(f'  [{lang}] REF: {s["sentence"]}')
    print(f'  [{lang}] HYP: {result}')
    seen.add(lang)
    if len(seen) == len(LANGUAGES): break

---
## Phase 5 - Gradio Demo App

Build an interactive demo inside Colab. Anyone can speak and see the transcription.

In [ ]:
# Cell 21 - Launch Gradio demo in Colab
import gradio as gr

def gradio_transcribe(audio):
    if audio is None: return 'Please record audio first!'
    sr, arr = audio
    return transcribe(arr, sr=sr) or '(No speech detected)'


demo = gr.Interface(
    fn=gradio_transcribe,
    inputs=gr.Audio(source='microphone', type='numpy',
                    label='Speak in Hindi | Telugu | Tamil | Kannada | English'),
    outputs=gr.Textbox(label='Transcription', lines=4),
    title='Multilingual ASR - 5 Indian Languages + English',
    description='Powered by wav2vec2-xls-r-300m fine-tuned on Mozilla Common Voice 11.0',
    allow_flagging='never',
    theme=gr.themes.Soft(),
)

demo.launch(share=True, server_name='0.0.0.0', server_port=7860)
print('Open the public URL above to use the demo!')

In [ ]:
# Cell 22 - Generate HuggingFace Spaces deployment files
os.makedirs('./hf_space', exist_ok=True)

app_content = '''import gradio as gr
import torch, numpy as np, librosa
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

SAMPLING_RATE = 16_000
MODEL_NAME    = "''' + MODEL_REPO_NAME + '''"
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor     = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model         = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

def transcribe(audio):
    if audio is None: return "Please record audio!"
    sr, arr = audio
    arr = arr.astype(np.float32)
    if arr.max() > 1.0: arr = arr / 32768.0
    if arr.ndim > 1: arr = arr.mean(axis=1)
    if sr != SAMPLING_RATE: arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLING_RATE)
    inp = processor(arr, sampling_rate=SAMPLING_RATE, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model(**{k: v.to(DEVICE) for k, v in inp.items()}).logits
    return processor.batch_decode(torch.argmax(logits, dim=-1))[0] or "(No speech)"

gr.Interface(
    fn=transcribe,
    inputs=gr.Audio(source="microphone", type="numpy"),
    outputs=gr.Textbox(label="Transcription", lines=3),
    title="Multilingual ASR - Hindi | Telugu | Tamil | Kannada | English",
    allow_flagging="never", theme=gr.themes.Soft(),
).launch()
'''

reqs = 'transformers>=4.30.0\ndatasets>=2.13.0\ntorch>=2.0.0\nlibrosa>=0.10.0\ngradio>=4.0.0\nsoundfile>=0.12.0\nhuggingface_hub>=0.16.0\nevaluate>=0.4.0\n'

with open('./hf_space/app.py', 'w') as f: f.write(app_content)
with open('./hf_space/requirements.txt', 'w') as f: f.write(reqs)
print('Deployment files saved to ./hf_space/')
print('Steps: huggingface.co/spaces > New Space > Gradio > Upload app.py + requirements.txt')

---
## Summary

| Phase | What we did |
|-------|-------------|
| 1 Setup | Install libs, HF login, configure project |
| 2 Hindi baseline | Data -> vocab -> processor -> train -> eval |
| 3 Multilingual | Repeat for 5 languages with unified vocab |
| 4 Evaluate | WER overall + per language; push to Hub |
| 5 Deploy | Gradio demo in Colab + HF Spaces files |

### References
- [wav2vec 2.0 paper](https://arxiv.org/abs/2006.11477)
- [XLS-R paper](https://arxiv.org/abs/2111.09296)
- [Mozilla Common Voice](https://commonvoice.mozilla.org/)
- [HF Wav2Vec2 docs](https://huggingface.co/docs/transformers/model_doc/wav2vec2)
